# 03 — Feature Engineering
**Where Should I Buy? · GeoAI Course**

Goals:
- Compute spatial distance features for each neighborhood
- Run the full scoring pipeline
- Inspect score distributions
- Save processed results to GeoParquet

In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.data.loaders import load_transactions, load_neighborhoods, load_transit_stops, load_parks
from src.data.preprocessing import (
    preprocess_transactions, preprocess_neighborhoods,
    preprocess_transit_stops, preprocess_parks
)
from src.geo.distance_features import compute_neighborhood_spatial_features
from src.geo.spatial_join import attach_transaction_stats_to_neighborhoods
from src.scoring.neighborhood_score import (
    ScoringWeights, run_scoring_pipeline, NeighborhoodScorer
)
from src.config import PROCESSED_NEIGHBORHOODS_PATH

# Load
transactions  = preprocess_transactions(load_transactions())
neighborhoods = preprocess_neighborhoods(load_neighborhoods())
transit_stops = preprocess_transit_stops(load_transit_stops())
parks         = preprocess_parks(load_parks())
print('Data loaded ✅')

## 1. Compute Spatial Features

In [ ]:
print('Computing spatial distance features...')
enriched = compute_neighborhood_spatial_features(neighborhoods.copy(), transit_stops, parks)

print('\nSpatial features added:')
spatial_cols = ['dist_to_nearest_transit_m', 'dist_to_nearest_park_m',
                'transit_stops_500m', 'transit_stops_1km', 'parks_1km']
enriched[spatial_cols].describe().round(1)

## 2. Attach Transaction Statistics

In [ ]:
enriched = attach_transaction_stats_to_neighborhoods(
    enriched, transactions, neighborhood_col='neighborhood'
)

print('Transaction columns added:')
tx_cols = ['avg_price', 'median_price', 'avg_price_per_m2', 'transaction_count']
enriched[tx_cols].describe().round(0)

## 3. Run Scoring Pipeline

In [ ]:
# Default weights
scored = run_scoring_pipeline(enriched)

# Show rankings
ranking_cols = ['name', 'rank', 'neighborhood_score',
                'score_affordability', 'score_transit', 'score_parks',
                'avg_price', 'transit_stops_500m', 'parks_1km']
available = [c for c in ranking_cols if c in scored.columns]
scored[available].sort_values('rank')

## 4. Sensitivity Analysis — Different Weight Scenarios

In [ ]:
scenarios = {
    'Default (40/35/25)':     ScoringWeights(0.40, 0.35, 0.25),
    'Price-focused (70/15/15)': ScoringWeights(0.70, 0.15, 0.15),
    'Transit-focused (20/60/20)': ScoringWeights(0.20, 0.60, 0.20),
    'Green-focused (20/20/60)': ScoringWeights(0.20, 0.20, 0.60),
}

results = {}
for name, weights in scenarios.items():
    scorer = NeighborhoodScorer(weights)
    s = scorer.fit_transform(enriched)
    results[name] = s.set_index('name')['neighborhood_score']

comparison = pd.DataFrame(results)
print('Neighborhood scores under different weight scenarios:')
comparison.round(1)

## 5. Score Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score bar chart
scored_sorted = scored.sort_values('neighborhood_score', ascending=True)
colors = ['#d73027' if s < 40 else '#fee08b' if s < 65 else '#1a9850'
          for s in scored_sorted['neighborhood_score']]
axes[0].barh(scored_sorted['name'], scored_sorted['neighborhood_score'],
             color=colors, alpha=0.85)
axes[0].axvline(50, color='gray', linestyle='--', alpha=0.5, label='Midpoint')
axes[0].set_xlabel('Suitability Score (0-100)')
axes[0].set_title('Neighborhood Rankings (Default Weights)', fontweight='bold')
axes[0].legend()

# Score components stacked
score_cols = ['score_affordability', 'score_transit', 'score_parks']
available_score_cols = [c for c in score_cols if c in scored.columns]
if available_score_cols:
    df_plot = scored.sort_values('neighborhood_score', ascending=False)
    x = range(len(df_plot))
    bottom = np.zeros(len(df_plot))
    palette = ['#4c72b0', '#dd8452', '#55a868']
    labels = ['Affordability', 'Transit', 'Parks']
    for col, color, label in zip(available_score_cols, palette, labels):
        vals = df_plot[col].values * 100
        axes[1].bar(x, vals, bottom=bottom, color=color, alpha=0.8, label=label)
        bottom += vals
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(df_plot['name'].tolist(), rotation=45, ha='right', fontsize=9)
    axes[1].set_ylabel('Sub-score contribution')
    axes[1].set_title('Score Components by Neighborhood', fontweight='bold')
    axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Save to GeoParquet

In [ ]:
PROCESSED_NEIGHBORHOODS_PATH.parent.mkdir(parents=True, exist_ok=True)
scored.to_parquet(PROCESSED_NEIGHBORHOODS_PATH)
print(f'✅ Saved scored neighborhoods to {PROCESSED_NEIGHBORHOODS_PATH}')
print(f'   Rows: {len(scored)}, Columns: {list(scored.columns)}')